# 03 — Phase 3: Resize + Pad for DINOv2

Takes accepted crops from `review_meta.json` (Phase 2 output), resizes with aspect ratio preserved,
detects the crop's background, and pads to **224 × 224** for DINOv2.
Then copies results to `final/` with the patent's first CPC code in each filename.

---

## Two Approaches, Side-by-Side

| | `legacy_padding_resize` | `adaptive_padding_resize` |
|---|---|---|
| **Background** | Fixed white `(255, 255, 255)` | Detected from outermost border pixels |
| **Gradient support** | ✗ — hard white border | ✓ — `BORDER_REFLECT_101` for variable backgrounds |
| **Dark crops** | Jarring white border | Matched to actual background |
| **Pre-sharpening** | ✗ | ✓ — 1.20× before downscale |

> **Prerequisites:** Phase 1 (extraction) and Phase 2 (review) must be complete.

Run cells top-to-bottom. Cells 4–5 are the visual + quantitative benchmark.
Cell 6 is the production run that writes to `processed/`.

In [ ]:
import sys; sys.path.insert(0, "..")
import json
import time
from pathlib import Path

import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image, ImageEnhance
from tqdm import tqdm

from src.config_loader import load_config
from src.processor import organize_processed

cfg = load_config()

CROPS_DIR     = Path(cfg["paths"]["crops"])
PROCESSED_DIR = Path(cfg["paths"]["processed"])
LOGS_DIR      = Path(cfg["paths"]["logs"])
TARGET_SIZE   = cfg["processor"]["target_size"]

print(f"Crops dir   : {CROPS_DIR}")
print(f"Processed   : {PROCESSED_DIR}")
print(f"Target size : {TARGET_SIZE} x {TARGET_SIZE} px")

In [ ]:
# Load accepted paths from Phase 2 review
meta_path = LOGS_DIR / "review_meta.json"
if not meta_path.exists():
    raise FileNotFoundError(
        f"review_meta.json not found at {meta_path}\n"
        "Run Phase 2 (02_review.ipynb) first."
    )

with open(meta_path) as f:
    meta = json.load(f)

accepted_paths = []
for patent_id, pdata in meta.items():
    if pdata.get("is_duplicate") or pdata.get("review_status") == "DISAPPROVED":
        continue
    for fname, idata in pdata.get("images", {}).items():
        if idata.get("approved") and not idata.get("needs_split"):
            p = CROPS_DIR / patent_id / fname
            if p.exists():
                accepted_paths.append(p)

print(f"Accepted crops to process : {len(accepted_paths)}")
print(f"Target size               : {TARGET_SIZE} x {TARGET_SIZE} px")
print(f"Processed output          : {PROCESSED_DIR}")
print(f"Final output (with CPC)   : {cfg['paths']['final']}")

# Demo fallback: if review hasn't been done yet, use a sample of raw crops
if not accepted_paths:
    accepted_paths = sorted(CROPS_DIR.rglob("*.png"))[:20]
    print(f"\n  (no review data found — using first {len(accepted_paths)} raw crops for demo)")

## Function Definitions

In [ ]:
def legacy_padding_resize(image: Image.Image, target_size: int = 224) -> Image.Image:
    """
    Resize + pad with a fixed white background.

    Ported from resize_with_padding() in Image_Chose_&_Save_PAdding_Enhanced_224x224.ipynb.
    Problem: always fills padding with (255, 255, 255) regardless of the actual crop
    background, creating jarring borders on dark, gray, or gradient images.
    """
    img = image.convert("RGB")
    w, h = img.size
    scale = target_size / max(w, h)
    new_w = max(1, int(round(w * scale)))
    new_h = max(1, int(round(h * scale)))

    resized  = img.resize((new_w, new_h), Image.Resampling.LANCZOS)
    canvas   = Image.new("RGB", (target_size, target_size), (255, 255, 255))
    offset_x = (target_size - new_w) // 2
    offset_y = (target_size - new_h) // 2
    canvas.paste(resized, (offset_x, offset_y))
    return canvas


def adaptive_padding_resize(image: Image.Image, target_size: int = 224) -> Image.Image:
    """
    Resize + pad with seamless, background-aware filling.

    Pipeline
    --------
    1. Sharpness 1.20x applied BEFORE downscaling — preserves high-frequency
       edge detail that would otherwise be blurred away by LANCZOS.
    2. Resize so the longest side = target_size (aspect ratio preserved).
    3. Sample the outermost `t` pixel rows/columns of the RESIZED image.
    4. Compute mean per-channel std dev across all border pixels:
       - std < 15  -> solid / near-solid background
                      -> fill with the median border color (clean, no smear)
       - std >= 15 -> gradient, dark translucent, or complex texture
                      -> use cv2.BORDER_REFLECT_101 to mirror edge content into
                         the padding, avoiding any hard color seam
    5. BORDER_REPLICATE fallback when padding exceeds image dimensions.

    Parameters
    ----------
    image       : PIL Image (any mode; converted to RGB internally)
    target_size : output square side length in pixels (224 for DINOv2)
    """
    img = image.convert("RGB")
    img = ImageEnhance.Sharpness(img).enhance(1.20)

    w, h = img.size
    scale = target_size / max(w, h)
    new_w = max(1, int(round(w * scale)))
    new_h = max(1, int(round(h * scale)))
    resized = img.resize((new_w, new_h), Image.Resampling.LANCZOS)

    pad_top    = (target_size - new_h) // 2
    pad_bottom = target_size - new_h - pad_top
    pad_left   = (target_size - new_w) // 2
    pad_right  = target_size - new_w - pad_left

    arr = np.array(resized, dtype=np.uint8)

    # Adaptive sample thickness: 1-3 px, scales with the shorter image dimension
    t = max(1, min(3, new_w // 20, new_h // 20))

    strips = [arr[:t, :].reshape(-1, 3), arr[-t:, :].reshape(-1, 3)]
    if new_h > 2 * t:
        strips += [arr[t:-t, :t].reshape(-1, 3), arr[t:-t, -t:].reshape(-1, 3)]
    border_pixels = np.vstack(strips)

    std_dev = border_pixels.std(axis=0).mean()

    if std_dev < 15.0:
        # Solid / near-solid background -> clean flat fill
        fill   = tuple(int(v) for v in np.median(border_pixels, axis=0))
        canvas = Image.new("RGB", (target_size, target_size), fill)
        canvas.paste(resized, (pad_left, pad_top))
        return canvas

    # Variable background -> mirror-extend edges (no hard seam)
    # BORDER_REFLECT_101 requires padding < image dimension on that axis
    can_reflect = (
        pad_top < new_h and pad_bottom < new_h and
        pad_left < new_w and pad_right < new_w
    )
    border_mode = cv2.BORDER_REFLECT_101 if can_reflect else cv2.BORDER_REPLICATE
    arr_padded  = cv2.copyMakeBorder(
        arr, pad_top, pad_bottom, pad_left, pad_right, border_mode
    )
    return Image.fromarray(arr_padded)


print("Defined: legacy_padding_resize | adaptive_padding_resize")

## Visual Comparison

Shows the first 8 accepted crops side-by-side: **Original → Legacy → Adaptive**.  
Look especially at rows with dark, gray, or gradient backgrounds — those show where legacy fails.

In [ ]:
N_PREVIEW    = min(8, len(accepted_paths))
sample_paths = accepted_paths[:N_PREVIEW]

fig, axes = plt.subplots(
    N_PREVIEW, 3,
    figsize=(10, N_PREVIEW * 3.5),
    squeeze=False,
)
fig.suptitle(
    "Padding Comparison  |  Original  ->  Legacy (white fill)  ->  Adaptive",
    fontsize=12, fontweight="bold", y=1.002,
)

for ax, title in zip(axes[0], ["Original (raw crop)", "Legacy (white fill)", "Adaptive (border-aware)"]):
    ax.set_title(title, fontsize=9, fontweight="bold")

for row_idx, crop_path in enumerate(sample_paths):
    img          = Image.open(crop_path).convert("RGB")
    legacy_out   = legacy_padding_resize(img, TARGET_SIZE)
    adaptive_out = adaptive_padding_resize(img, TARGET_SIZE)

    for col_idx, (panel, label) in enumerate([
        (img,          f"{crop_path.name[:40]}\n{img.width}x{img.height}"),
        (legacy_out,   f"{TARGET_SIZE}x{TARGET_SIZE}"),
        (adaptive_out, f"{TARGET_SIZE}x{TARGET_SIZE}"),
    ]):
        ax = axes[row_idx][col_idx]
        ax.imshow(panel)
        ax.set_xlabel(label, fontsize=6)
        ax.set_xticks([]); ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)

plt.tight_layout()
plt.show()

## Quantitative Benchmark

**Border seam std** measures the mean per-channel standard deviation of the outermost 3 px
of the *output* 224×224 image. A lower value means the padding blends smoothly into the
content — a better result for the model.

In [ ]:
def _border_seam_std(img224: Image.Image, t: int = 3) -> float:
    """Mean per-channel std dev of the outermost t pixels — proxy for seam harshness."""
    arr    = np.array(img224, dtype=np.float32)
    h, w   = arr.shape[:2]
    strips = [
        arr[:t, :].reshape(-1, 3),
        arr[-t:, :].reshape(-1, 3),
        arr[t:-t, :t].reshape(-1, 3),
        arr[t:-t, -t:].reshape(-1, 3),
    ]
    return float(np.vstack(strips).std(axis=0).mean())


BENCH_N = min(50, len(accepted_paths))
bench_paths = accepted_paths[:BENCH_N]

legacy_times,   adaptive_times   = [], []
legacy_seams,   adaptive_seams   = [], []

for p in tqdm(bench_paths, desc="Benchmarking"):
    img = Image.open(p).convert("RGB")

    t0 = time.perf_counter()
    l_out = legacy_padding_resize(img, TARGET_SIZE)
    legacy_times.append(time.perf_counter() - t0)
    legacy_seams.append(_border_seam_std(l_out))

    t0 = time.perf_counter()
    a_out = adaptive_padding_resize(img, TARGET_SIZE)
    adaptive_times.append(time.perf_counter() - t0)
    adaptive_seams.append(_border_seam_std(a_out))

print(f"\n{'Metric':<32} {'Legacy':>10} {'Adaptive':>10}")
print("-" * 54)
print(f"{'Mean time (ms)':<32} {np.mean(legacy_times)*1000:>9.1f} {np.mean(adaptive_times)*1000:>9.1f}")
print(f"{'Mean border seam std':<32} {np.mean(legacy_seams):>9.2f} {np.mean(adaptive_seams):>9.2f}")
print(f"{'Median border seam std':<32} {np.median(legacy_seams):>9.2f} {np.median(adaptive_seams):>9.2f}")
n_wins = sum(a < l for a, l in zip(adaptive_seams, legacy_seams))
print(f"{'Crops where adaptive wins':<32} {n_wins:>9} / {BENCH_N}")

## Production Run

Applies `adaptive_padding_resize` to every accepted crop and writes to `processed/`.
Already-processed files are skipped — safe to re-run.

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

processed_paths = []
skipped = 0
errors  = 0

for crop_path in tqdm(accepted_paths, desc="Processing crops"):
    record_id  = crop_path.name.split("_p")[0]
    patent_out = PROCESSED_DIR / record_id
    patent_out.mkdir(parents=True, exist_ok=True)
    out_path   = patent_out / crop_path.name

    if out_path.exists():
        processed_paths.append(out_path)
        skipped += 1
        continue

    try:
        img    = Image.open(crop_path).convert("RGB")
        result = adaptive_padding_resize(img, TARGET_SIZE)
        result.save(out_path, "PNG")
        processed_paths.append(out_path)
    except Exception as e:
        print(f"  Error processing {crop_path.name}: {e}")
        errors += 1

print(f"\nDone. {len(processed_paths) - skipped} new | {skipped} skipped | {errors} errors")
print(f"Output: {PROCESSED_DIR}")

In [ ]:
# Quick sanity check — show first 6 processed images
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, p in zip(axes.flat, processed_paths[:6]):
    ax.imshow(Image.open(p))
    ax.set_title(Path(p).name[:35], fontsize=7)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Copy to final/ with CPC code injected into each filename
# Output: final/{patent_id}/{patent_id}_{CPC}_p{page}_c{crop}.png
# CPC codes are read from the PatSeer Excel (paths.excel)
organize_processed(cfg)